# Notebook 02 · Recuperación top-k, construcción de contexto e integración con LLM en MongoDB Atlas

Este notebook continúa el flujo del notebook 01, pero ahora sobre **MongoDB Atlas Vector Search**:

- recuperación `top-k`
- construcción de contexto
- integración con un LLM
- persistencia de consultas y respuestas
- comparación de estrategias de chunking

## 1. Instalación de dependencias

In [ ]:
!pip -q install pymongo[srv] sentence-transformers openai pandas nltk

## 2. Importaciones y configuración inicial

In [ ]:
import os
import json
import time
import getpass
import textwrap
import warnings
from typing import List, Dict, Optional

import pandas as pd
import nltk
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer
from openai import OpenAI

warnings.filterwarnings("ignore")
nltk.download("punkt", quiet=True)

print("[OK] Librerías del sistema CookFlow cargadas correctamente.")

[OK] Librerías del sistema CookFlow cargadas correctamente.


## 3. Variables de conexión

In [ ]:
# 1. Credenciales desde secretos de Colab
from google.colab import userdata
MONGODB_URI = userdata.get('MONGO_URI')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# 2. Configuración de colecciones
DB_NAME = "cookflow"
CHUNKS_COLLECTION_NAME = "chunks_embeddings"
HISTORY_COLLECTION_NAME = "auditoria_rag"

# 3. Configuración del LLM
LLM_BASE_URL = "https://api.groq.com/openai/v1"
LLM_MODEL = "llama-3.3-70b-versatile"

# 4. Parametros del modelo de embeddings e indice vectorial
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
VECTOR_INDEX_NAME = "vector_index_texto"

print("[OK] Configuracion cargada desde secretos de Colab.")

[OK] Configuracion cargada desde secretos de Colab.


## 4. Conexión a MongoDB Atlas y carga del modelo MiniLM

In [ ]:
# Conexión al cliente NoSQL utilizando los nombres oficiales
client = MongoClient(MONGODB_URI)
db = client[DB_NAME]

chunks_collection = db[CHUNKS_COLLECTION_NAME]
history_collection = db[HISTORY_COLLECTION_NAME]

# Carga del modelo de embeddings (all-MiniLM-L6-v2)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("=== ESTADO DE CONEXIÓN OFICIAL EN ATLAS ===")
print("Base de datos:", DB_NAME)
print(f" -> Colección '{CHUNKS_COLLECTION_NAME}': {chunks_collection.count_documents({})} fragmentos vectorizados.")
print(f" -> Colección '{HISTORY_COLLECTION_NAME}': {history_collection.count_documents({})} interacciones registradas.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

=== ESTADO DE CONEXIÓN OFICIAL EN ATLAS ===
Base de datos: cookflow
 -> Colección 'chunks_embeddings': 242 fragmentos vectorizados.
 -> Colección 'auditoria_rag': 0 interacciones registradas.


## 5. Verificación rápida de colecciones

Este notebook asume que el notebook 01 ya cargó chunks en la colección principal.

In [ ]:
status_rows = []
# Validamos las colecciones principales del sistema corporativo
colecciones_a_revisar = ['recetas', 'chunks_embeddings', 'ingredientes', 'auditoria_rag', 'valoraciones']

for name in colecciones_a_revisar:
    try:
        status_rows.append({
            "Colección CookFlow": name,
            "Documentos Actuales": db[name].count_documents({})
        })
    except Exception as e:
        status_rows.append({
            "Colección CookFlow": name,
            "Documentos Actuales": f"ERROR: {e}"
        })

pd.DataFrame(status_rows)

,Colección CookFlow,Documentos Actuales
0,recetas,50
1,chunks_embeddings,242
2,ingredientes,50
3,auditoria_rag,0
4,valoraciones,35


## 6. Funciones de Búsqueda Semántica y Construcción de Contexto

In [ ]:
def embed_query(question: str) -> List[float]:
    return embedding_model.encode(question, normalize_embeddings=True).tolist()

def get_llm_client(api_key: str, base_url: str = LLM_BASE_URL):
    if not api_key:
        raise ValueError("No se proporcionó API key para el LLM.")
    return OpenAI(api_key=api_key, base_url=base_url)

def answer_with_llm(
    client: OpenAI,
    question: str,
    context: str,
    model: str = LLM_MODEL,
    temperature: float = 0.2,
    max_tokens: int = 700
) -> str:
    system_prompt = (
        "Eres CookFlow-AI, un asistente experto en alta cocina y gastronomia. "
        "Debes responder unicamente con base en el contexto recuperado. "
        "Si el contexto no es suficiente, dilo explicitamente. "
        "No inventes datos, no cites fuentes inexistentes y se muy preciso."
    )
    user_prompt = f"""
Pregunta del usuario:
{question}

Contexto recuperado:
{context}

Instrucciones:
- Responde en español de forma clara.
- Cuando sea util, menciona el titulo de la receta de la que proviene el contexto.
"""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

def build_context(
    retrieved_df: pd.DataFrame,
    max_chars: int = 5000,
    include_metadata: bool = True
) -> str:
    if retrieved_df is None or retrieved_df.empty:
        return ""
    blocks = []
    total_chars = 0
    for i, row in retrieved_df.iterrows():
        titulo     = row.get('titulo_receta', 'Receta CookFlow')
        tipo       = row.get('tipo_fuente', 'N/A')
        estrategia = row.get('estrategia_chunking', 'N/A')
        score      = row.get('score', 0.0)
        texto      = row.get('texto_chunk', '')
        if include_metadata:
            block = (
                f"[Fragmento {i+1}]\n"
                f"Receta: {titulo}\n"
                f"Tipo: {tipo}\n"
                f"Estrategia: {estrategia}\n"
                f"Score: {float(score):.4f}\n"
                f"Contenido:\n{texto}"
            )
        else:
            block = f"[Fragmento {i+1}]\n{texto}"
        if total_chars + len(block) > max_chars:
            break
        blocks.append(block)
        total_chars += len(block) + 2
    return "\n\n".join(blocks)

def search_top_k(
    collection,
    question_vector: List[float],
    top_k: int = 5,
    strategy: Optional[str] = None,
    tipo_fuente: Optional[str] = None,
    dificultad: Optional[str] = None,
    max_calorias: Optional[int] = None,
    max_tiempo: Optional[int] = None,
    idioma: Optional[str] = None,
    num_candidates: int = 150
) -> pd.DataFrame:
    filters = []

    if strategy:
        if strategy in ["fixed", "A"]:   strategy_mapeada = "fixed-size"
        elif strategy in ["sentence", "B"]: strategy_mapeada = "sentence-aware"
        elif strategy in ["semantic", "C"]: strategy_mapeada = "semantic"
        else: strategy_mapeada = strategy
        filters.append({"estrategia_chunking": {"$eq": strategy_mapeada}})

    if tipo_fuente:
        filters.append({"tipo_fuente": {"$eq": tipo_fuente}})

    # Filtros hibridos sobre campo meta
    if dificultad:
        filters.append({"meta.dificultad": {"$eq": dificultad}})
    if max_calorias:
        filters.append({"meta.calorias": {"$lte": max_calorias}})
    if max_tiempo:
        filters.append({"meta.tiempo": {"$lte": max_tiempo}})
    if idioma:
        filters.append({"meta.idioma": {"$eq": idioma}})

    vector_stage = {
        "$vectorSearch": {
            "index": VECTOR_INDEX_NAME,
            "path": "embedding",
            "queryVector": question_vector,
            "numCandidates": max(num_candidates, top_k),
            "limit": top_k
        }
    }
    if filters:
        vector_stage["$vectorSearch"]["filter"] = {"$and": filters} if len(filters) > 1 else filters[0]

    pipeline = [
        vector_stage,
        {
            "$project": {
                "_id": 0,
                "id_documento_fuente": 1,
                "titulo_receta": 1,
                "tipo_fuente": 1,
                "chunk_id": 1,
                "estrategia_chunking": 1,
                "texto_chunk": 1,
                "meta": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
    ]
    return pd.DataFrame(list(collection.aggregate(pipeline)))

## 7. Persistencia del Historial (Ingesta de Logs RAG)

In [ ]:
def ensure_logging_indexes():
    history_collection.create_index("fecha")
    history_collection.create_index("id_consulta")
    history_collection.create_index("tipo_registro")


def save_query_and_results(
    question: str,
    retrieved_df: pd.DataFrame,
    answer: Optional[str] = None,
    metadata: Optional[dict] = None
) -> str:
    ensure_logging_indexes()
    metadata = metadata or {}

    import uuid
    id_consulta = str(uuid.uuid4())
    timestamp_actual = pd.Timestamp.utcnow().isoformat()

    retrieval_docs = []
    if retrieved_df is not None and not retrieved_df.empty:
        for rank, row in enumerate(retrieved_df.to_dict(orient="records"), start=1):
            retrieval_docs.append({
                "ranking": rank,
                "doc_id": row.get("id_documento_fuente"),
                "chunk_index": row.get("chunk_index"),
                "titulo": row.get("titulo_receta"),
                "estrategia_chunking": row.get("estrategia_chunking"),
                "score": float(row.get("score", 0.0)),
                "texto_chunk": row.get("texto_chunk"),
            })

    # Guardamos toda la interacción de auditoría de forma estructurada en auditoria_rag
    registro_completo = {
        "id_consulta": id_consulta,
        "fecha": timestamp_actual,
        "tipo_registro": "auditoria_rag_cookflow",
        "usuario_id": "anonimo_chef",
        "interaccion": {
            "question": question,
            "chunks_recuperados": retrieval_docs,
            "answer": answer if answer else "N/A"
        },
        "metadata": metadata
    }

    history_collection.insert_one(registro_completo)
    return id_consulta

## 8. Pipeline RAG Consolidado

In [ ]:
def rag_pipeline(
    question: str,
    top_k: int = 5,
    strategy: Optional[str] = None,
    tipo_fuente: Optional[str] = None,
    max_context_chars: int = 5000,
    use_llm: bool = True
) -> Dict:
    qvec = embed_query(question)

    retrieved = search_top_k(
        collection=chunks_collection,
        question_vector=qvec,
        top_k=top_k,
        strategy=strategy,
        tipo_fuente=tipo_fuente
    )

    context = build_context(
        retrieved_df=retrieved,
        max_chars=max_context_chars,
        include_metadata=True
    )

    answer = None
    if use_llm:
        if not context.strip():
            answer = "No se recuperó contexto suficiente para responder la pregunta."
        else:
            client_llm = get_llm_client(GROQ_API_KEY, LLM_BASE_URL)
            answer = answer_with_llm(client_llm, question, context)

    query_id = save_query_and_results(
        question=question,
        retrieved_df=retrieved,
        answer=answer,
        metadata={
            "top_k": top_k,
            "strategy": strategy,
            "tipo_fuente": tipo_fuente
        }
    )

    return {
        "id_consulta": query_id,
        "question": question,
        "retrieved": retrieved,
        "context": context,
        "answer": answer
    }

## 9. Celda de Prueba de Recuperación Semántica

In [ ]:
# Solicitamos la pregunta gastronómica en tiempo real al usuario
pregunta_usuario = input("¿Qué deseas buscar en el asistente de cocina CookFlow?: ").strip()

if pregunta_usuario:
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.width', 1000)
    # Ejecutamos la búsqueda vectorial usando tus funciones oficiales
    df_test = search_top_k(
        collection=chunks_collection,
        question_vector=embed_query(pregunta_usuario),
        top_k=5
    )

    print(f"\nResultados principales recuperados en Atlas para: '{pregunta_usuario}'")
    display(df_test)
else:
    print("Operación cancelada. No ingresaste ninguna consulta.")

¿Qué deseas buscar en el asistente de cocina CookFlow?: pollo al curry

Resultados principales recuperados en Atlas para: 'pollo al curry'


,id_documento_fuente,chunk_id,estrategia_chunking,meta,texto_chunk,tipo_fuente,titulo_receta,score
0,664a1b2c3d4e5f6a7b8c9f03,664a1b2c3d4e5f6a7b8c9f03_prep_0,sentence-aware,"{'dificultad': 'facil', 'calorias': 0, 'tags': ['pollo', 'curry', 'asiatica', 'sin-gluten'], 'idioma': 'es', 'tiempo': 30}","Preparación de Pollo al Curry Express: Cortar el pollo en cubos medianos Picar la cebolla y el ajo finamente y sofreir en aceite hasta transparente Agregar el pollo y dorar por 5 minutos Anadir el curry en polvo y mezclar bien para activar los aromas Verter la leche de coco, bajar el fuego y dejar reducir 15 minutos Rectificar sal y pimienta antes de servir con arroz basmati",receta,Pollo al Curry Express,0.786214
1,664a1b2c3d4e5f6a7b8c9f03,664a1b2c3d4e5f6a7b8c9f03_ing_0,fixed-size,"{'dificultad': 'facil', 'calorias': 0, 'tags': ['pollo', 'curry', 'asiatica', 'sin-gluten'], 'idioma': 'es', 'tiempo': 30}","Ingredientes de Pollo al Curry Express: 500 gr Pechuga de pollo, 200 ml Leche de coco, 2 cda Curry en polvo, 1 unidad Cebolla larga, 2 diente Ajo",receta,Pollo al Curry Express,0.764526
2,664a1b2c3d4e5f6a7b8c9f28,664a1b2c3d4e5f6a7b8c9f28_ing_0,fixed-size,"{'dificultad': 'media', 'calorias': 0, 'tags': ['india', 'pollo', 'especiada', 'sin-gluten'], 'idioma': 'es', 'tiempo': 50}","Ingredientes de Pollo Tikka Masala: 600 gr Pechuga de pollo, 400 gr Tomates maduros, 15 gr Curry en polvo, 4 diente Ajo",receta,Pollo Tikka Masala,0.735982
3,nota_664a1b2c3d4e5f6a7b8c9c32,nota_664a1b2c3d4e5f6a7b8c9c32_nota_0,semantic,NaN,Nota de Carlos Mendez para la receta 664a1b2c3d4e5f6a7b8c9f39: El pollo al ajillo con dos cabezas de ajo enteras suena exagerado pero el resultado es sublime.,nota_cocinero,Nota de Carlos Mendez,0.723848
4,664a1b2c3d4e5f6a7b8c9f17,664a1b2c3d4e5f6a7b8c9f17_ing_0,fixed-size,"{'dificultad': 'media', 'calorias': 0, 'tags': ['peruana', 'pollo', 'horneado', 'sin-gluten'], 'idioma': 'es', 'tiempo': 100}","Ingredientes de Pollo a la Brasa Peruano: 1 unidad Pollo entero, 30 gr Aji panca, 6 diente Ajo",receta,Pollo a la Brasa Peruano,0.709441


## 10. Previsualización del Contexto Consolidado

In [ ]:
# Construimos el bloque de contexto usando la tabla de resultados df_test generada arriba
context_preview = build_context(df_test, max_chars=2200, include_metadata=True)
print("=== CONTEXTO CONSOLIDADO ENVIADO AL LLM ===")
print(context_preview)

=== CONTEXTO CONSOLIDADO ENVIADO AL LLM ===
[Fragmento 1]
Receta: Pollo al Curry Express
Tipo: receta
Estrategia: sentence-aware
Score: 0.7862
Contenido:
Preparación de Pollo al Curry Express: Cortar el pollo en cubos medianos Picar la cebolla y el ajo finamente y sofreir en aceite hasta transparente Agregar el pollo y dorar por 5 minutos Anadir el curry en polvo y mezclar bien para activar los aromas Verter la leche de coco, bajar el fuego y dejar reducir 15 minutos Rectificar sal y pimienta antes de servir con arroz basmati

[Fragmento 2]
Receta: Pollo al Curry Express
Tipo: receta
Estrategia: fixed-size
Score: 0.7645
Contenido:
Ingredientes de Pollo al Curry Express: 500 gr Pechuga de pollo, 200 ml Leche de coco, 2 cda Curry en polvo, 1 unidad Cebolla larga, 2 diente Ajo

[Fragmento 3]
Receta: Pollo Tikka Masala
Tipo: receta
Estrategia: fixed-size
Score: 0.7360
Contenido:
Ingredientes de Pollo Tikka Masala: 600 gr Pechuga de pollo, 400 gr Tomates maduros, 15 gr Curry en polvo, 4 die

## 11. Generación de la Respuesta con el LLM

In [ ]:
if GROQ_API_KEY:
    client_llm = get_llm_client(GROQ_API_KEY, LLM_BASE_URL)
    # Enviamos de forma dinámica la 'pregunta_usuario' que ingresaste por pantalla
    answer_preview = answer_with_llm(
        client=client_llm,
        question=pregunta_usuario,
        context=context_preview
    )
    print("\n=== RESPUESTA GENERADA POR COOKFLOW-AI ===")
    print(answer_preview)
else:
    print("No se proporcionó GROQ_API_KEY. Este bloque se omite.")


=== RESPUESTA GENERADA POR COOKFLOW-AI ===
El pollo al curry es un plato delicioso y fácil de preparar. Según la receta "Pollo al Curry Express", para prepararlo debes seguir los siguientes pasos:

1. Cortar el pollo en cubos medianos.
2. Picar la cebolla y el ajo finamente y sofreír en aceite hasta que estén transparentes.
3. Agregar el pollo y dorar durante 5 minutos.
4. Añadir el curry en polvo y mezclar bien para activar los aromas.
5. Verter la leche de coco, bajar el fuego y dejar reducir durante 15 minutos.
6. Rectificar la sal y la pimienta antes de servir con arroz basmati.

En cuanto a los ingredientes, la receta "Pollo al Curry Express" requiere:
- 500 gr de pechuga de pollo
- 200 ml de leche de coco
- 2 cucharadas de curry en polvo
- 1 cebolla larga
- 2 dientes de ajo

Espero que esta información te sea útil. ¡Disfruta preparando y comiendo este delicioso pollo al curry!


## 12. Ejecución del Pipeline RAG Completo

In [ ]:
pregunta_pipeline = input("Ingresa tu consulta para CookFlow: ").strip()

if pregunta_pipeline:
    result = rag_pipeline(
        question=pregunta_pipeline,
        top_k=5,
        strategy=None,
        tipo_fuente=None,
        max_context_chars=4500,
        use_llm=bool(GROQ_API_KEY)
    )
    print("\nID consulta:", result["id_consulta"])
    print("\n=== RESPUESTA GENERADA POR COOKFLOW-AI ===")
    print(result["answer"] if result["answer"] else "[Sin llamada al LLM]")
else:
    print("Operacion cancelada.")

Ingresa tu consulta para CookFlow: receta con pollo

ID consulta: 2b144918-564b-43c9-afe6-d5e05b914537

=== RESPUESTA GENERADA POR COOKFLOW-AI ===
La receta que se ajusta a tu solicitud es el "Pollo a la Brasa Peruano". A continuación, te proporciono los ingredientes que se mencionan en el contexto recuperado:

* 1 unidad de pollo entero
* 30 gr de Aji panca
* 6 dientes de ajo

Es importante destacar que el contexto recuperado no proporciona instrucciones detalladas para preparar el "Pollo a la Brasa Peruano", pero sí menciona algunos consejos y notas de cocineros que podrían ser útiles. Por ejemplo, una nota de Sofia Ramirez sugiere marinar el pollo en yogur antes de cocinarlo.

Si deseas obtener más información o instrucciones detalladas para preparar el "Pollo a la Brasa Peruano", te recomiendo buscar una receta completa que incluya todos los pasos y técnicas necesarias.


## 13. Inspección de los Chunks Utilizados en el Pipeline anterior

In [ ]:
# Despliega el DataFrame con los fragmentos de soporte de la consulta anterior
result["retrieved"]

,id_documento_fuente,chunk_id,estrategia_chunking,meta,texto_chunk,tipo_fuente,titulo_receta,score
0,664a1b2c3d4e5f6a7b8c9f17,664a1b2c3d4e5f6a7b8c9f17_ing_0,fixed-size,"{'dificultad': 'media', 'calorias': 0, 'tags': ['peruana', 'pollo', 'horneado', 'sin-gluten'], 'idioma': 'es', 'tiempo': 100}","Ingredientes de Pollo a la Brasa Peruano: 1 unidad Pollo entero, 30 gr Aji panca, 6 diente Ajo",receta,Pollo a la Brasa Peruano,0.796755
1,nota_664a1b2c3d4e5f6a7b8c9c05,nota_664a1b2c3d4e5f6a7b8c9c05_nota_1,semantic,NaN,Todos los invitados pidieron la receta.,nota_cocinero,Nota de Ana Jimenez,0.796132
2,nota_664a1b2c3d4e5f6a7b8c9c32,nota_664a1b2c3d4e5f6a7b8c9c32_nota_0,semantic,NaN,Nota de Carlos Mendez para la receta 664a1b2c3d4e5f6a7b8c9f39: El pollo al ajillo con dos cabezas de ajo enteras suena exagerado pero el resultado es sublime.,nota_cocinero,Nota de Carlos Mendez,0.780794
3,nota_664a1b2c3d4e5f6a7b8c9c17,nota_664a1b2c3d4e5f6a7b8c9c17_nota_0,semantic,NaN,Nota de Valentina Cruz para la receta 664a1b2c3d4e5f6a7b8c9f13: El guacamole rustico para la reunion fue un exito.,nota_cocinero,Nota de Valentina Cruz,0.780665
4,nota_664a1b2c3d4e5f6a7b8c9c02,nota_664a1b2c3d4e5f6a7b8c9c02_nota_0,semantic,NaN,Nota de Sofia Ramirez para la receta 664a1b2c3d4e5f6a7b8c9f03: Segui el consejo de marinar el pollo en yogur antes.,nota_cocinero,Nota de Sofia Ramirez,0.780244


## 14. Monitoreo y Auditoría del Historial en auditoria_rag

In [ ]:
print("=== ÚLTIMAS CONSULTAS REGISTRADAS EN COOKFLOW ===")
# Extraemos las preguntas recientes guardadas en la estructura de auditoría unificada
consultas_recientes = list(history_collection.find(
    {"tipo_registro": "auditoria_rag_cookflow"},
    {"interaccion.question": 1, "fecha": 1, "metadata": 1}
).sort("fecha", -1).limit(5))

# Formateamos el resultado de forma plana para que la tabla de Pandas sea legible
df_consultas = pd.DataFrame([{
    "Fecha UTC": c.get("fecha"),
    "Pregunta del Usuario": c.get("interaccion", {}).get("question"),
    "Top_K": c.get("metadata", {}).get("top_k")
} for c in consultas_recientes])
display(df_consultas)


print("\n=== ÚLTIMOS RESULTADOS Y RESPUESTAS DEL LLM ===")
# Extraemos los fragmentos y las respuestas completas de la misma colección oficial
resultados_recientes = list(history_collection.find(
    {"tipo_registro": "auditoria_rag_cookflow"},
    {"id_consulta": 1, "interaccion.question": 1, "interaccion.answer": 1, "fecha": 1}
).sort("fecha", -1).limit(5))

df_resultados = pd.DataFrame([{
    "ID Consulta": r.get("id_consulta"),
    "Fecha UTC": r.get("fecha"),
    "Pregunta": r.get("interaccion", {}).get("question"),
    "Respuesta IA (Primeros 150 carácteres)": r.get("interaccion", {}).get("answer")[:150] + "..." if r.get("interaccion", {}).get("answer") else "N/A"
} for r in resultados_recientes])
display(df_resultados)

=== ÚLTIMAS CONSULTAS REGISTRADAS EN COOKFLOW ===


,Fecha UTC,Pregunta del Usuario,Top_K
0,2026-06-10T19:28:01.051929+00:00,receta con pollo,5



=== ÚLTIMOS RESULTADOS Y RESPUESTAS DEL LLM ===


,ID Consulta,Fecha UTC,Pregunta,Respuesta IA (Primeros 150 carácteres)
0,2b144918-564b-43c9-afe6-d5e05b914537,2026-06-10T19:28:01.051929+00:00,receta con pollo,"La receta que se ajusta a tu solicitud es el ""Pollo a la Brasa Peruano"". A continuación, te proporciono los ingredientes que se mencionan en el contex..."


## 15. Comparación de estrategias de chunking

In [ ]:
# EJEMPLOS RECOMENDADOS CULINARIOS PARA PROBAR LA COMPARATIVA:
# - ¿Cómo preparar el pollo al curry express?
# - ¿Qué consejos hay sobre el uso de la leche de coco?

# El usuario ingresa la pregunta para evaluar cómo rinde cada estrategia
question_compare = input("Ingresa una pregunta para comparar el score de las 3 estrategias de chunking: ").strip()

if question_compare:
    comparison = []

    # Iteramos de forma limpia por tus 3 estrategias de corte
    for strategy in ["fixed-size", "sentence-aware", "semantic"]:
        tmp = search_top_k(
            collection=chunks_collection,
            question_vector=embed_query(question_compare),
            top_k=3,
            strategy=strategy
        )
        if not tmp.empty:
            comparison.append({
                "Estrategia Evaluada": strategy,
                "Top 1 Score (Coseno)": float(tmp.iloc[0]["score"]),
                "Receta Origen": tmp.iloc[0]["titulo_receta"], # Nombre real de tu columna
                "Primeros 120 caracteres": tmp.iloc[0]["texto_chunk"][:120] + "..." # Nombre real de tu columna
            })

    # Desplegamos la tabla comparativa de rendimiento para el profesor
    display(pd.DataFrame(comparison))
else:
    print("Operación cancelada. No ingresaste ninguna consulta para comparar.")

## 16. Análisis estadístico de chunks por estrategia

In [ ]:
print("=== DISTRIBUCIÓN DE CHUNKS EN ATLAS POR ESTRATEGIA ===")

pipeline_stats = [
    {"$group": {
        "_id": "$estrategia_chunking",
        "cantidad": {"$sum": 1},
        "longitud_promedio": {"$avg": "$n_chars"},
        "longitud_min": {"$min": "$n_chars"},
        "longitud_max": {"$max": "$n_chars"}
    }},
    {"$sort": {"_id": 1}}
]

stats = list(chunks_collection.aggregate(pipeline_stats))
df_stats = pd.DataFrame(stats).rename(columns={"_id": "estrategia_chunking"})
df_stats["longitud_promedio"] = df_stats["longitud_promedio"].round(1)
display(df_stats)

## 17. Función de Atajo ask() y Ejemplo Interactivo

In [ ]:
def ask(question: str, top_k: int = 5, strategy: Optional[str] = None, tipo_fuente: Optional[str] = None):
    # Llama al pipeline consolidado que creamos en los bloques anteriores
    result = rag_pipeline(
        question=question,
        top_k=top_k,
        strategy=strategy,
        tipo_fuente=tipo_fuente,
        use_llm=bool(GROQ_API_KEY)
    )
    print("\n" + "="*60)
    print("🤖 RESPUESTA EN DIRECTO DESDE ASK():")
    print("="*60)
    print(result["answer"] if result["answer"] else "[Sin llamada al LLM]")
    print("="*60 + "\n")
    return result

# =====================================================================
# EJECUCIÓN DEL EJEMPLO FINAL INTERACTIVO USANDO LA FUNCIÓN ASK()
# =====================================================================
pregunta_ask = input("Haz una consulta rápida usando la función simplificada ask(): ").strip()

if pregunta_ask:
    r = ask(pregunta_ask, top_k=5, tipo_fuente="receta")
else:
    print("No ingresaste ninguna pregunta para la función ask().")